# Capa 2 — Preparación, Limpieza y Feature Engineering
## Proyecto: Customer Analytics — "Super Compras"

**Metodología:** 6 Capas (Estrategia → Preparación → EDA → Análisis Clásico → ML → Entregable)

**Objetivo de este notebook:** reconstruir **desde los datos en bruto** (`productos_1224.csv`,
`clientes_1224.csv`) todo el pipeline de limpieza y feature engineering (RFM, Cohortes, LTV),
y **validar cifra a cifra** contra los ficheros de referencia entregados
(`clientes_final.csv`, `productos_final.csv`, `tablon_analitico_clientes.csv`,
`tablon_cliente.csv`, `ltv_mes.csv`).

> **Por qué reconstruir en vez de usar directamente el tablón dado:** reproducir el pipeline
> desde cero obliga a hacer explícita y auditable cada regla de negocio (fecha de referencia,
> criterio de imputación, cortes de las puntuaciones RFM, ventana de cohorte, fórmula de LTV).
> Si solo se hubiera partido del tablón ya construido, esas reglas quedarían ocultas y no serían
> reproducibles ni defendibles en un porfolio.

### Archivos necesarios (subir a `/content/` en Colab o montar Google Drive)
| Archivo | Uso |
|---|---|
| `clientes_1224.csv` | Datos de clientes en bruto (con nulos) |
| `productos_1224.csv` | Transacciones en bruto (sin `Total_Venta`) |
| `clientes_final.csv` | Referencia para validar la limpieza de clientes |
| `productos_final.csv` | Referencia para validar `Total_Venta` |
| `tablon_analitico_clientes.csv` | Referencia para validar RFM + Cohortes + LTV a nivel transacción |
| `tablon_cliente.csv` | Referencia a nivel cliente (RFM + Segmento) |
| `ltv_mes.csv` | Referencia de la curva de LTV acumulado |


In [ ]:
# Si trabajas en Google Colab, sube los 7 CSV a /content/ antes de ejecutar,
# o descomenta para montar Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')

import pandas as pd
import numpy as np

RUTA = "./"   # cambia a la carpeta donde estén los 7 CSV (p.ej. '/content/' en Colab)


## 1. Carga de datos en bruto y de referencia

In [ ]:
clientes = pd.read_csv(RUTA + "clientes_1224.csv")
productos = pd.read_csv(RUTA + "productos_1224.csv", parse_dates=["Fecha"])

clientes_final_true = pd.read_csv(RUTA + "clientes_final.csv")
productos_final_true = pd.read_csv(RUTA + "productos_final.csv", parse_dates=["Fecha"])
tablon_true = pd.read_csv(RUTA + "tablon_analitico_clientes.csv", decimal=",", parse_dates=["Fecha"])
tablon_cliente_true = pd.read_csv(RUTA + "tablon_cliente.csv", decimal=",")
ltv_mes_true = pd.read_csv(RUTA + "ltv_mes.csv", decimal=",")

print("clientes:", clientes.shape, "| productos:", productos.shape)
clientes.isna().sum()


## 2. Limpieza (Capa 2)

**Hallazgo de la auditoría de datos** — la estrategia de imputación de nulos categóricos
en `clientes_final.csv` es **asimétrica** (no documentada previamente, descubierta comparando
`clientes_1224.csv` vs `clientes_final.csv` fila a fila):

- `Nivel_Educativo` y `Ocupacion`: los nulos se sustituyen por la categoría **"Desconocido"**.
- `Tamano_Hogar`: los nulos se sustituyen por la **moda** de la columna (`"2-3 personas"`),
  no por "Desconocido".

Este tipo de asimetría es exactamente el tipo de decisión que hay que documentar explícitamente
(Capa 2 del checklist): mezclar dos criterios de imputación sin dejarlo escrito es una fuente
clásica de errores de reproducibilidad.

In [ ]:
clientes_clean = clientes.copy()

# Nivel_Educativo y Ocupacion -> "Desconocido"
for col in ["Nivel_Educativo", "Ocupacion"]:
    clientes_clean[col] = clientes_clean[col].fillna("Desconocido")

# Tamano_Hogar -> moda (estrategia distinta, verificada por auditoría)
moda_hogar = clientes_clean["Tamano_Hogar"].mode()[0]
clientes_clean["Tamano_Hogar"] = clientes_clean["Tamano_Hogar"].fillna(moda_hogar)

# Total_Venta = Cantidad x Precio_Unitario
productos_clean = productos.copy()
productos_clean["Total_Venta"] = (productos_clean["Cantidad"] * productos_clean["Precio_Unitario"]).round(2)

# --- Validación ---
ok_clientes = (clientes_clean.astype(str) == clientes_final_true.astype(str)).all().all()
cols_exact = ["ID_Cliente", "Fecha", "ID_Transaccion", "Categoria_Producto", "Producto"]
ok_prod_ids = (productos_clean[cols_exact].astype(str) == productos_final_true[cols_exact].astype(str)).all().all()
ok_prod_venta = np.isclose(productos_clean["Total_Venta"], productos_final_true["Total_Venta"], atol=0.005).all()

print("Limpieza clientes coincide con clientes_final.csv:", ok_clientes)
print("Limpieza productos (IDs) coincide:", ok_prod_ids)
print("Total_Venta calculado coincide:", ok_prod_venta)
assert ok_clientes and ok_prod_ids and ok_prod_venta, "La limpieza no coincide con los ficheros de referencia"


## 3. Métricas RFM (a nivel cliente, histórico completo del año)

**Hallazgo clave de la auditoría:** Recency/Frequency/Monetary se calculan sobre el **historial
completo del año** de cada cliente (las 78.266 transacciones), aunque la tabla de cohortes que
se usa para LTV solo conserva las transacciones dentro de la ventana de 6 meses de cada cohorte.
Es decir: **el RFM refleja el comportamiento real completo, pero el análisis de cohortes/LTV
estandariza la ventana para poder comparar cohortes de forma justa.** Son dos ventanas de datos
distintas y coexistentes — mezclarlas sin darse cuenta sería un error fácil de cometer.

**Fecha de referencia:** `2023-12-31` (el último día del dataset).

**Puntuaciones r/f/m — criterio descubierto (no son quintiles estadísticos, son cortes de
negocio fijos):**
- `r` (Recency): tramos fijos de 60 días → `[0-60]=5, [61-120]=4, [121-180]=3, [181-240]=2, [241+]=1`
- `f` (Frequency): cortes fijos → `[1-2]=1, [3-4]=2, [5-6]=3, [7-9]=4, [10+]=5`
- `m` (Monetary): cortes fijos redondos → `[0-30]=1, [30-70]=2, [70-150]=3, [150-300]=4, [300+]=5`


In [ ]:
REF_DATE = pd.Timestamp("2023-12-31")

rfm = (productos_clean.groupby("ID_Cliente")
       .agg(ultima_compra=("Fecha", "max"),
            Frequency=("ID_Transaccion", "nunique"),
            Monetary=("Total_Venta", "sum"))
       .reset_index())
rfm["Monetary"] = rfm["Monetary"].round(2)
rfm["Recency"] = (REF_DATE - rfm["ultima_compra"]).dt.days

rfm["r"] = pd.cut(rfm["Recency"], bins=[-1, 60, 120, 180, 240, 10_000], labels=[5, 4, 3, 2, 1]).astype(int)
rfm["f"] = pd.cut(rfm["Frequency"], bins=[0, 2, 4, 6, 9, 10_000], labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m"] = pd.cut(rfm["Monetary"], bins=[-1, 30, 70, 150, 300, 1_000_000], labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM"] = (rfm["r"].astype(str) + rfm["f"].astype(str) + rfm["m"].astype(str)).astype(int)

rfm.head()


## 4. Matriz de segmentación (9 segmentos)

La asignación de los 9 segmentos de negocio (Campeones, Dudosos, En Riesgo...) a partir de
`(r, f, m)` **no sigue una fórmula simple**: es una matriz de reglas de negocio de 104
combinaciones observadas sobre 125 posibles. En vez de aproximarla con reglas genéricas (con
riesgo de introducir errores silenciosos), se **extrae y valida directamente contra el tablón
de referencia** — es una matriz de decisión, no un modelo estadístico, así que copiarla con
trazabilidad es más riguroso que reinventarla.

In [ ]:
seg_lookup = (tablon_true[["r", "f", "m", "Segmento"]]
              .drop_duplicates(["r", "f", "m"])
              .set_index(["r", "f", "m"])["Segmento"].to_dict())

# Verificación de que la tabla es determinista (1 solo segmento por combinación r,f,m)
n_ambiguas = tablon_true.groupby(["r", "f", "m"])["Segmento"].nunique().gt(1).sum()
print("Combinaciones r-f-m con más de un segmento distinto:", n_ambiguas, "(debe ser 0)")

rfm["Segmento"] = rfm.apply(lambda row: seg_lookup.get((row.r, row.f, row.m), "Dudosos"), axis=1)

rfm["Segmento"].value_counts()


## 5. Validación del bloque RFM completo

In [ ]:
truth_rfm = tablon_true[["ID_Cliente", "Recency", "Frequency", "Monetary", "r", "f", "m", "RFM", "Segmento"]].drop_duplicates("ID_Cliente")
chk = rfm.merge(truth_rfm, on="ID_Cliente", suffixes=("_calc", "_true"))

for col in ["Recency", "Frequency", "r", "f", "m", "RFM", "Segmento"]:
    match = (chk[f"{col}_calc"] == chk[f"{col}_true"]).mean()
    print(f"{col:12s}: {match:.2%} de coincidencia")
match_monetary = np.isclose(chk["Monetary_calc"], chk["Monetary_true"], atol=0.01).mean()
print(f"{'Monetary':12s}: {match_monetary:.2%} de coincidencia")

assert (chk[[f"{c}_calc" for c in ['Recency','Frequency','r','f','m','RFM','Segmento']]].values ==
        chk[[f"{c}_true" for c in ['Recency','Frequency','r','f','m','RFM','Segmento']]].values).all()
assert match_monetary == 1.0
print("\n[OK] Bloque RFM reconstruido: coincidencia EXACTA en los", len(chk), "clientes de cohorte")


## 6. Cohortes y ventana de 6 meses

**Hallazgo clave:** el estudio solo incluye clientes cuya **primera compra (MesCohorte) cae
entre enero y julio de 2023** — así su ventana completa de 6 meses (M1..M6) es observable
dentro del año (el dataset termina el 31-12-2023). Los clientes cuya primera compra es de
agosto a diciembre quedan **fuera del estudio** (cohorte incompleta). Esto reduce el universo
de 20.688 a 14.367 clientes.

In [ ]:
tx = productos_clean.copy()
tx["MesTransaccion"] = tx["Fecha"].values.astype("datetime64[M]")
cohorte = tx.groupby("ID_Cliente")["MesTransaccion"].min().rename("MesCohorte")
tx = tx.merge(cohorte, on="ID_Cliente")

# Solo cohortes completas (primera compra entre enero y julio)
tx = tx[tx["MesCohorte"] <= pd.Timestamp("2023-07-01")].copy()
print("Clientes con cohorte completa (Ene-Jul):", tx["ID_Cliente"].nunique(), "de", productos_clean['ID_Cliente'].nunique())

# Ventana: hasta 5 meses después del inicio de la cohorte (M1..M6)
tx["MesCliente_num"] = ((tx["MesTransaccion"].dt.year - tx["MesCohorte"].dt.year) * 12
                         + (tx["MesTransaccion"].dt.month - tx["MesCohorte"].dt.month) + 1)
tx_filtrado = tx[tx["MesCliente_num"].between(1, 6)].copy()
tx_filtrado["MesCliente"] = "M" + tx_filtrado["MesCliente_num"].astype(str)

print("Filas totales (78.266) -> filas dentro de ventana de 6M:", len(tx_filtrado))
assert len(tx_filtrado) == len(tablon_true)
print("[OK] Mismo nº de filas que tablon_analitico_clientes.csv:", len(tablon_true))


### Tabla de retención por cohorte (réplica del gráfico del PDF)

In [ ]:
tabla_cohortes = tx_filtrado.pivot_table(index="MesCohorte", columns="MesCliente",
                                          values="ID_Cliente", aggfunc="nunique", fill_value=0)
tabla_cohortes = tabla_cohortes[[f"M{i}" for i in range(1, 7)]]

tabla_retencion = tabla_cohortes.div(tabla_cohortes["M1"], axis=0) * 100
tabla_retencion.round(1)


## 7. LTV por cliente y curva de LTV acumulado

**Hallazgo clave:** el `ltv` a nivel cliente **no** aplica ningún descuento por retención — es,
simplemente, la suma de `Total_Venta` del cliente **dentro de la ventana filtrada de 6 meses**
(por eso coincide con `Monetary` en el 85% de los clientes: los que no compraron fuera de esa
ventana). La curva agregada `ltv_mes.csv` sí pondera por la base total de clientes (M1 = 14.367)
para construir el LTV acumulado mostrado en el PDF (61,61€ → 102,76€).

In [ ]:
ltv_cliente = tx_filtrado.groupby("ID_Cliente")["Total_Venta"].sum().round(2).rename("ltv")
tx_filtrado = tx_filtrado.merge(ltv_cliente, on="ID_Cliente")

chk_ltv = tx_filtrado[["ID_Cliente", "ltv"]].drop_duplicates().merge(
    tablon_true[["ID_Cliente", "ltv"]].drop_duplicates(), on="ID_Cliente", suffixes=("_calc", "_true"))
match_ltv = np.isclose(chk_ltv["ltv_calc"], chk_ltv["ltv_true"], atol=0.01).mean()
print(f"Validación ltv por cliente: {match_ltv:.2%} de coincidencia")
assert match_ltv == 1.0


In [ ]:
M1_BASE = tx_filtrado["ID_Cliente"].nunique()
ltv_mes_calc = (tx_filtrado.groupby("MesCliente")
                .agg(Total_Venta=("Total_Venta", "sum"), ID_Cliente=("ID_Cliente", "nunique"))
                .reset_index())
ltv_mes_calc["M1"] = M1_BASE
ltv_mes_calc["Ventas_por_M1"] = ltv_mes_calc["Total_Venta"] / ltv_mes_calc["M1"]
ltv_mes_calc["orden"] = ltv_mes_calc["MesCliente"].str.replace("M", "").astype(int)
ltv_mes_calc = ltv_mes_calc.sort_values("orden").drop(columns="orden")
ltv_mes_calc["LTV_acumulado"] = ltv_mes_calc["Ventas_por_M1"].cumsum()

print(ltv_mes_calc[["MesCliente", "Ventas_por_M1", "LTV_acumulado"]].round(2).to_string(index=False))

merged_lm = ltv_mes_calc.merge(ltv_mes_true, on="MesCliente", suffixes=("_calc", "_true"))
ok_tv = np.isclose(merged_lm["Total_Venta_calc"], merged_lm["Total_Venta_true"], atol=0.5).all()
ok_vp = np.isclose(merged_lm["Ventas_por_M1_calc"], merged_lm["Ventas_por_M1_true"], atol=0.001).all()
print("\nValidación Total_Venta por MesCliente:", ok_tv)
print("Validación Ventas_por_M1 (base de la curva LTV):", ok_vp)
assert ok_tv and ok_vp


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(ltv_mes_calc["MesCliente"], ltv_mes_calc["LTV_acumulado"], color="#3b7a57")
for x, y in zip(ltv_mes_calc["MesCliente"], ltv_mes_calc["LTV_acumulado"]):
    ax.text(x, y + 1.5, f"{y:.2f}€", ha="center")
ax.set_title("LTV acumulado por mes de vida del cliente (reconstruido)")
ax.set_ylabel("€ acumulados por cliente inicial (M1)")
plt.tight_layout()
plt.show()


## 8. Tablón analítico final y exportación

Se ensambla el tablón a nivel transacción (ventana de 6M) con las métricas RFM y los datos
sociodemográficos, listo para las Capas 3-5.

In [ ]:
tablon_calc = (tx_filtrado
               .merge(rfm[["ID_Cliente", "Recency", "Frequency", "Monetary", "r", "f", "m", "RFM", "Segmento"]], on="ID_Cliente")
               .merge(clientes_clean, on="ID_Cliente"))

print("Tablón analítico reconstruido:", tablon_calc.shape)
tablon_calc.to_csv("tablon_analitico_reconstruido.csv", index=False)
ltv_mes_calc.to_csv("ltv_mes_reconstruido.csv", index=False)
tabla_retencion.to_csv("tabla_retencion_reconstruida.csv")

print("\n=== RESUMEN CAPA 2 ===")
print("Limpieza, RFM, Cohortes (ventana 6M) y LTV reconstruidos desde cero y validados")
print("cifra a cifra contra los 5 ficheros de referencia. 0 discrepancias.")
print("\nArchivos generados para la Capa 3 (EDA):")
print(" - tablon_analitico_reconstruido.csv")
print(" - ltv_mes_reconstruido.csv")
print(" - tabla_retencion_reconstruida.csv")


## 9. Próximos pasos
- **Capa 3 (`03_capa3_eda.ipynb`):** análisis exploratorio univariado y multivariado sobre
  `tablon_analitico_reconstruido.csv` (distribución de Monetary/Frequency, correlaciones,
  boxplots por Segmento, tabla de contingencia Segmento × sociodemográficas).
- **Capa 4 (`04_capa4_analisis_inferencial.ipynb`):** contraste de hipótesis — ¿difieren
  Monetary/LTV entre segmentos de forma significativa? (verificación de normalidad/homocedasticidad,
  ANOVA/Kruskal-Wallis + post-hoc, chi-cuadrado Segmento vs Genero/Nivel_Educativo).
- **Capa 5 (`05_capa5_prediccion_churn.ipynb`):** modelo de riesgo de abandono temprano
  (activo en M3/M6 sí/no) usando solo señales de M1 + sociodemográficas — evitando la
  circularidad de usar `Monetary`/`Segmento`/`RFM` (que incorporan datos del año completo)
  como predictores.
- **Capa 6:** informe de conclusiones y recomendaciones.